# Phase 3 -- Session-5 2x2 cross-format evaluation (frozen, inference-only)

Final untouched-Session-5 evaluation. Loads the two ALREADY-TRAINED adapters
(`adapter_best` from the completed NoHist and Hist3 full training runs) and evaluates
each against BOTH Session-5 formats, on the identical frozen 1622-utterance six-class
set used throughout this project (`kaggle_prep/kaggle_upload/test.json` /
`test_history3.json`):

  A = NoHist-adapter x NoHist-eval   (matched)
  B = NoHist-adapter x Hist3-eval    (mismatched)
  C = Hist3-adapter  x NoHist-eval   (mismatched)
  D = Hist3-adapter  x Hist3-eval    (matched)

This notebook does **no training, no optimizer, no checkpoint selection, no early
stopping, and no adaptation of any kind based on Session 5** -- every adapter's weights
were already fixed by the Session-4 validation in the training runs, before this
notebook ever touches Session 5. Designed to run start-to-finish in one
Save Version -> Save & Run All.


In [1]:
# ============================ CONFIG ============================
MODEL_ID = "Qwen/Qwen2.5-Omni-3B"
SEED = 11
MAX_NEW_TOKENS = 12   # identical generation setting to every prior notebook in this project
OUTPUT_DIR = "/kaggle/working/phase3_s5_2x2eval"

# Manual override only if auto-discovery (below) fails or is ambiguous -- leave as None to
# auto-discover from whatever Kaggle Input datasets/notebook-outputs are attached.
NOHIST_ADAPTER_DIR_OVERRIDE = None
HIST3_ADAPTER_DIR_OVERRIDE = None

EVAL_LIB_SHA256_EXPECTED = "24fac1f49d6f049c1485b4769ce523a292ae692fad4657a9fab7be6af6034298"

print("SCOPE: Session-5 2x2 cross-format evaluation. Inference only. No training.")
print("OUTPUT_DIR:", OUTPUT_DIR)


SCOPE: Session-5 2x2 cross-format evaluation. Inference only. No training.
OUTPUT_DIR: /kaggle/working/phase3_s5_2x2eval


In [2]:
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=4.52.3", "accelerate>=0.34", "qwen-omni-utils", "librosa", "soundfile", "peft>=0.11.1", "scikit-learn")
# NOTE: deliberately NOT reinstalling numpy/scipy (see kaggle_phase3_smoketest.ipynb /
# kaggle_phase3_train.ipynb history -- force-reinstalling them to "latest" produced an
# internally-inconsistent numpy install on this image). Kaggle's own pre-installed numpy/scipy
# pairing is used as-is. scikit-learn IS installed explicitly -- iemocap_eval_lib.py genuinely
# needs it for real (accuracy_score/f1_score/etc.), unlike transformers' own OPTIONAL sklearn
# import path.
import numpy as _np_check, scipy as _sp_check
print("numpy  :", _np_check.__version__, "->", _np_check.__file__)
print("scipy  :", _sp_check.__version__, "->", _sp_check.__file__)
# Kaggle's base image ships a stale torchao (0.10.0) that peft's LoRA module-dispatch chain
# unconditionally probes when wrapping ANY target module (even though we never use torchao/
# quantization) -- peft's is_torchao_available() raises rather than skipping on a too-old
# install. Purely an environment fix; does not enable or touch quantization.
pip("-U", "torchao>=0.16.0")
print("pip done")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 32.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.67.0 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.67.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.

numpy  : 2.0.2 -> /usr/local/lib/python3.12/dist-packages/numpy/__init__.py
scipy  : 1.16.3 -> /usr/local/lib/python3.12/dist-packages/scipy/__init__.py
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 38.3 MB/s eta 0:00:00
pip done


In [3]:
import os, json, time, random, hashlib, base64
import numpy as np
# numpy/scipy private-API compat shim (same fix required in Experiment 4's Kaggle image)
import numpy._core._multiarray_umath as _mu
if not hasattr(_mu, "_blas_supports_fpe"):
    _mu._blas_supports_fpe = lambda *a, **k: False

# numpy.char / numpy.strings / numpy._core.strings / numpy._core.defchararray have shown THREE
# different, independent internal breakages across different Kaggle sessions with this SAME
# notebook code (missing compiled ufunc names; a missing __all__ on a half-initialized module;
# a real ufunc object that doesn't support __module__ assignment on this numpy build) -- each a
# different entry point, each only discovered after fixing the previous one. Rather than keep
# patching individual symptoms, unconditionally replace all four with inert placeholder modules
# BEFORE anything (torch/transformers/scipy/peft) can trigger any of them. We never use any
# numpy.char/numpy.strings functionality anywhere in this pipeline (audio, tokenization, JSON
# only), so short-circuiting them entirely -- regardless of whether this particular session's
# numpy happens to be broken -- is safe and removes this whole class of failure for good.
import sys, types


def _inert_numpy_string_module(name):
    m = types.ModuleType(name)
    m.__all__ = []
    m.__doc__ = f"Inert placeholder for {name} (unused in this pipeline)."
    return m


for _mod_name in ("numpy._core.strings", "numpy._core.defchararray", "numpy.char", "numpy.strings"):
    sys.modules[_mod_name] = _inert_numpy_string_module(_mod_name)
np.char = sys.modules["numpy.char"]
np.strings = sys.modules["numpy.strings"]
print("numpy.char/numpy.strings/numpy._core.strings/numpy._core.defchararray preemptively "
      "replaced with inert placeholders (never used in this pipeline) -- avoids this Kaggle "
      "image's numpy string-ufunc internals entirely, regardless of which entry point would "
      "otherwise trigger them.")
import torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import transformers, peft
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("CUDA        :", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}, "
          f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB")
DTYPE = torch.float16  # T4s -> fp16, matching every prior notebook in this project
os.makedirs(OUTPUT_DIR, exist_ok=True)


numpy.char/numpy.strings/numpy._core.strings/numpy._core.defchararray preemptively replaced with inert placeholders (never used in this pipeline) -- avoids this Kaggle image's numpy string-ufunc internals entirely, regardless of which entry point would otherwise trigger them.


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


torch       : 2.10.0+cu128
transformers: 5.17.0
peft        : 0.21.0
CUDA        : True | GPUs: 2
  cuda:0 = Tesla T4, 15.6 GB
  cuda:1 = Tesla T4, 15.6 GB


In [4]:
# ---- locate the Session-5 dataset (kaggle_upload/: test.json + test_history3.json + audio/) ----
# Recursive walk (not a one-level os.listdir check) -- Kaggle can nest an extracted zip's
# contents one level deeper than the dataset root (e.g. /kaggle/input/<name>/kaggle_upload/
# test.json), which a shallow check misses. Same robust pattern as find_adapter_dir() below.
def _find_s5_input_dir():
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if "test.json" in files and "test_history3.json" in files:
            hits.append(root)
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        raise AssertionError(
            f"Found multiple candidate Session-5 directories under /kaggle/input: {hits} -- "
            f"ambiguous, remove the extra one(s) and re-run.")
    return None


S5_INPUT_DIR = _find_s5_input_dir()
if S5_INPUT_DIR is None:
    print("Could not find test.json + test_history3.json anywhere under /kaggle/input.")
    print("Current /kaggle/input contents (up to 3 levels deep), for debugging:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        if depth > 3:
            dirs[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{root}/")
        for fn in files[:10]:
            print(f"{indent}  {fn}")
        if len(files) > 10:
            print(f"{indent}  ... ({len(files)} files total)")
    raise AssertionError(
        "Could not find the Session-5 dataset (needs test.json AND test_history3.json "
        "somewhere under /kaggle/input, at any depth). Add the kaggle_upload/ dataset as a "
        "Kaggle Input, then re-run. See the directory listing printed above for what IS "
        "currently attached.")
S5_AUDIO_DIR = os.path.join(S5_INPUT_DIR, "audio")
assert os.path.isdir(S5_AUDIO_DIR), f"audio/ not found under {S5_INPUT_DIR}"
print("S5_INPUT_DIR:", S5_INPUT_DIR)
print("S5_AUDIO_DIR:", S5_AUDIO_DIR, "|", len(os.listdir(S5_AUDIO_DIR)), "wav files")


# ---- locate the two adapter_best checkpoints from the completed training runs ----
# Add each completed training notebook's OUTPUT as a Kaggle Input ("Add Input" -> search your
# own notebooks -> select the committed nohist/hist3 full-run notebook). Auto-discovered by
# walking /kaggle/input for a directory ending in "phase3_train_<variant>_full/adapter_best" --
# robust to whatever exact mount path Kaggle gives the attached notebook output.
def find_adapter_dir(variant, override):
    if override:
        assert os.path.isdir(override), f"{variant} override path does not exist: {override}"
        return override
    marker = os.path.join(f"phase3_train_{variant}_full", "adapter_best")
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if root.endswith(marker):
            hits.append(root)
    assert len(hits) >= 1, (
        f"Could not find a '.../{marker}' directory anywhere under /kaggle/input -- did you "
        f"Add Input the completed '{variant}' training run's notebook output? "
        f"(Or set {variant.upper()}_ADAPTER_DIR_OVERRIDE in the CONFIG cell.)")
    assert len(hits) == 1, (
        f"Found multiple candidates for the {variant} adapter: {hits} -- ambiguous, "
        f"set {variant.upper()}_ADAPTER_DIR_OVERRIDE in the CONFIG cell to disambiguate.")
    return hits[0]


NOHIST_ADAPTER_DIR = find_adapter_dir("nohist", NOHIST_ADAPTER_DIR_OVERRIDE)
HIST3_ADAPTER_DIR = find_adapter_dir("hist3", HIST3_ADAPTER_DIR_OVERRIDE)
print("NOHIST_ADAPTER_DIR:", NOHIST_ADAPTER_DIR)
print("  contents:", sorted(os.listdir(NOHIST_ADAPTER_DIR)))
print("HIST3_ADAPTER_DIR:", HIST3_ADAPTER_DIR)
print("  contents:", sorted(os.listdir(HIST3_ADAPTER_DIR)))


S5_INPUT_DIR: /kaggle/input/datasets/pranavjaiganesh13/dataset1
S5_AUDIO_DIR: /kaggle/input/datasets/pranavjaiganesh13/dataset1/audio | 1622 wav files
NOHIST_ADAPTER_DIR: /kaggle/input/notebooks/pranavjaiganesh13/nohist-run/phase3_train_nohist_full/adapter_best
  contents: ['README.md', 'adapter_config.json', 'adapter_model.safetensors']
HIST3_ADAPTER_DIR: /kaggle/input/notebooks/pranavjaiganesh13/hist3-run/phase3_train_hist3_full/adapter_best
  contents: ['README.md', 'adapter_config.json', 'adapter_model.safetensors']


In [5]:
# ---- HF auth (optional; Qwen is ungated) ----
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    if _tok:
        from huggingface_hub import login
        login(token=_tok)
        print("HF login OK")
except Exception as e:
    print("no HF_TOKEN secret / not needed for this ungated model:", repr(e))


HF login OK


In [6]:
# ================= iemocap_eval_lib.py, embedded verbatim as base64 =================
_EVAL_LIB_B64 = """..."""
_eval_lib_bytes = base64.b64decode(_EVAL_LIB_B64)
_actual_sha = hashlib.sha256(_eval_lib_bytes).hexdigest()
assert _actual_sha == EVAL_LIB_SHA256_EXPECTED, (
    f"iemocap_eval_lib.py sha256 mismatch: expected {EVAL_LIB_SHA256_EXPECTED}, got {_actual_sha}")
with open("/kaggle/working/iemocap_eval_lib.py", "wb") as f:
    f.write(_eval_lib_bytes)
sys.path.insert(0, "/kaggle/working")
import iemocap_eval_lib as EVAL
print("iemocap_eval_lib.py OK, sha256 =", _actual_sha[:16] + "...")
print("IEMOCAP_LABELS:", EVAL.IEMOCAP_LABELS)


iemocap_eval_lib.py OK, sha256 = 24fac1f49d6f049c...
IEMOCAP_LABELS: ['happy', 'sad', 'neutral', 'angry', 'excited', 'frustrated']


In [7]:
# ================= load the frozen Session-5 NoHist/Hist3 sets + audit =================
def _load(fn):
    with open(os.path.join(S5_INPUT_DIR, fn), encoding="utf-8") as f:
        return json.load(f)

nohist_s5 = _load("test.json")
hist3_s5 = _load("test_history3.json")

EXPECTED_N_S5 = 1622

# ---- audit: identical target utterances/order across both formats, before any model is loaded ----
assert len(nohist_s5) == EXPECTED_N_S5, f"expected {EXPECTED_N_S5} NoHist S5 rows, got {len(nohist_s5)}"
assert len(hist3_s5) == EXPECTED_N_S5, f"expected {EXPECTED_N_S5} Hist3 S5 rows, got {len(hist3_s5)}"
nohist_ids = [r["id"] for r in nohist_s5]
hist3_ids = [r["id"] for r in hist3_s5]
assert nohist_ids == hist3_ids, "NoHist/Hist3 S5 id order diverged -- STOP, not a valid paired design"
assert len(set(nohist_ids)) == EXPECTED_N_S5, "duplicate ids in the S5 set"
assert all("history_context" not in r for r in nohist_s5), "NoHist file unexpectedly carries history_context"
assert all("history_context" in r for r in hist3_s5), "Hist3 file missing history_context"
sessions = {r["session"] for r in nohist_s5}
assert sessions == {5}, f"expected Session 5 only, got sessions {sessions} -- wrong dataset attached"
golds_n = [r["output"] for r in nohist_s5]
golds_h = [r["output"] for r in hist3_s5]
assert golds_n == golds_h, "gold labels diverged between NoHist/Hist3 S5 files -- STOP"
assert set(golds_n) == set(EVAL.IEMOCAP_LABELS), f"label set diverged: {set(golds_n)}"

import collections
print(f"Session-5 frozen set: {len(nohist_s5)} rows, identical ids/order/golds between "
      f"NoHist and Hist3 formats (audited above), all Session {sorted(sessions)}")
print("per-class:", dict(collections.Counter(golds_n)))
print("SHARED_S5_TARGET_IDS will be used as the common id list for every one of the 4 runs below.")
SHARED_S5_TARGET_IDS = nohist_ids


Session-5 frozen set: 1622 rows, identical ids/order/golds between NoHist and Hist3 formats (audited above), all Session [5]
per-class: {'neutral': 384, 'frustrated': 381, 'angry': 170, 'sad': 245, 'excited': 299, 'happy': 143}
SHARED_S5_TARGET_IDS will be used as the common id list for every one of the 4 runs below.


In [8]:
# ================= fresh-base-model loader + inference-mode adapter loading =================
# Re-derives TARGET_MODULES dynamically from the actual loaded model's named_modules(), using
# the EXACT collision-safe logic validated in kaggle_phase3_smoketest.ipynb / kaggle_phase3_train.ipynb.
import re
from transformers import Qwen2_5OmniForConditionalGeneration as OmniCls
from peft import PeftModel

EXPECTED_N_TARGET_MODULES = 144
EXPECTED_N_ADAPTER_PARAMS = 7_372_800


def derive_target_modules(base_model):
    proj_matches = [n for n, m in base_model.named_modules()
                    if isinstance(m, torch.nn.Linear) and re.search(r"\.(q|k|v|o)_proj$", n)]
    THINKER_LLM_PREFIX = "thinker.model."
    target_modules = sorted(n for n in proj_matches if n.startswith(THINKER_LLM_PREFIX))
    assert target_modules, "no thinker.model.* q/k/v/o_proj modules found -- architecture drift?"
    bad = [n for n in target_modules if any(seg in n for seg in
                                             ["audio_tower", "visual", "talker", "token2wav"])]
    assert not bad, f"target list still includes non-LLM pathway modules: {bad}"
    assert len(target_modules) == EXPECTED_N_TARGET_MODULES, (
        f"expected {EXPECTED_N_TARGET_MODULES} target modules, got {len(target_modules)} "
        f"-- architecture/checkpoint drift, STOP")
    return target_modules


def load_fresh_base_model():
    """Always loads a GENUINELY FRESH base model from the original checkpoint -- never
    reuses an already-PEFT-mutated object (same discipline as training: never wrap an
    already-peft-tainted base object a second time)."""
    return OmniCls.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")


def load_adapter_for_eval(base_model, adapter_dir):
    """Inference-only load: is_trainable=False (no gradients, no optimizer, nothing trainable
    -- this notebook never trains anything). Structural audit only: confirms the correct,
    complete adapter (144 target modules, exactly the expected parameter count) was loaded,
    and that NOTHING is trainable (n_trainable == 0), rather than asserting requires_grad==True
    the way the training notebook's audit does."""
    target_modules = derive_target_modules(base_model)
    model = PeftModel.from_pretrained(base_model, adapter_dir, is_trainable=False)

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n_trainable == 0, (
        f"expected 0 trainable params in inference-only mode, got {n_trainable} -- STOP, "
        f"this notebook must never train or adapt based on Session 5")

    not_lora = [tm for tm in target_modules
                if not (hasattr(model.get_submodule(tm), "lora_A") and
                        hasattr(model.get_submodule(tm), "lora_B"))]
    assert not not_lora, f"target modules missing lora_A/lora_B: {not_lora[:5]}"

    adapter_param_count = sum(p.numel() for n, p in model.named_parameters()
                              if "lora_" in n and "thinker.model." in n)
    assert adapter_param_count == EXPECTED_N_ADAPTER_PARAMS, (
        f"expected exactly {EXPECTED_N_ADAPTER_PARAMS} LoRA parameters from {adapter_dir}, "
        f"got {adapter_param_count} -- wrong/corrupt checkpoint?")

    model.eval()
    print(f"  loaded adapter from {adapter_dir}: {adapter_param_count:,} LoRA params across "
          f"{len(target_modules)} target modules, 0 trainable (inference-only), model.eval()")
    return model


In [9]:
# ================= processor + prompt construction (validated pattern, reused) =================
from transformers import AutoProcessor
proc = AutoProcessor.from_pretrained(MODEL_ID)

QWEN_SYS = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
            "capable of perceiving auditory and visual inputs, as well as generating text and speech.")


def build_generation_inputs(rec, history_context):
    """Prompt-only (no answer appended) -- identical construction to the training notebook's
    build_generation_inputs / the smoke-test's Gate-7 pattern."""
    prompt_text = EVAL.build_prompt(rec["utterance"], history_context)
    wav_path = os.path.join(S5_AUDIO_DIR, rec["id"] + ".wav")
    conv = [
        {"role": "system", "content": [{"type": "text", "text": QWEN_SYS}]},
        {"role": "user", "content": [
            {"type": "audio", "audio": wav_path},
            {"type": "text", "text": prompt_text}]},
    ]
    from qwen_omni_utils import process_mm_info
    rendered = proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    try:
        audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
    except TypeError:
        audios, images, videos = process_mm_info(conv)
    try:
        prompt_inputs = proc(text=rendered, audio=audios, images=images, videos=videos,
                             return_tensors="pt", padding=False)
    except TypeError:
        prompt_inputs = proc(text=rendered, audios=audios, images=images, videos=videos,
                             return_tensors="pt", padding=False)
    return prompt_inputs


def to_model_device(inputs, ref_model):
    first_param = next(ref_model.parameters())
    device, dtype = first_param.device, first_param.dtype
    moved = {}
    for k, v in inputs.items():
        if torch.is_tensor(v):
            v = v.to(device)
            if torch.is_floating_point(v):
                v = v.to(dtype)
            moved[k] = v
        else:
            moved[k] = v
    return moved


preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

In [10]:
# ================= inference-only evaluation (adapted from run_validation, no train() switch) =================
def run_s5_eval(model, thinker, records, tag):
    """Pure inference: top-level model.generate(...) (the validated, working path), under
    torch.no_grad(), model.eval() throughout -- no model.train() switch-back anywhere in this
    notebook, since nothing here ever trains. Saves predictions + metrics under OUTPUT_DIR,
    tagged by (adapter, eval-format) rather than by epoch."""
    thinker.config.use_cache = True
    model.eval()
    raw_answers = []
    t0 = time.time()
    with torch.no_grad():
        for i, rec in enumerate(records):
            history_context = rec.get("history_context")
            inputs = build_generation_inputs(rec, history_context)
            inputs = to_model_device(inputs, ref_model=model)
            try:
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     return_audio=False)
            except TypeError:
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            if isinstance(out, (tuple, list)):
                out = out[0]
            gen = out[:, inputs["input_ids"].shape[1]:]
            raw = proc.batch_decode(gen, skip_special_tokens=True)[0].strip()
            raw_answers.append(raw)
            if (i + 1) % 200 == 0:
                elapsed = time.time() - t0
                print(f"    {tag}: {i+1}/{len(records)}  ({elapsed:.0f}s elapsed, "
                      f"{elapsed/(i+1):.2f}s/example)")

    metrics = EVAL.score_predictions_normalized(records, raw_answers)

    preds_path = os.path.join(OUTPUT_DIR, f"s5_predictions_{tag}.json")
    with open(preds_path + ".tmp", "w", encoding="utf-8") as f:
        json.dump(metrics["predictions"], f, ensure_ascii=False, indent=2)
    os.replace(preds_path + ".tmp", preds_path)
    metrics_path = os.path.join(OUTPUT_DIR, f"s5_metrics_{tag}.json")
    with open(metrics_path + ".tmp", "w", encoding="utf-8") as f:
        json.dump({k: v for k, v in metrics.items() if k not in ("predictions",)},
                   f, ensure_ascii=False, indent=2, default=str)
    os.replace(metrics_path + ".tmp", metrics_path)

    print(f"  S5[{tag}]: WA={metrics['accuracy_WA']}  UA={metrics['UA_unweighted_recall']}  "
          f"macro_f1={metrics['macro_f1']}  weighted_f1={metrics['weighted_f1']}  "
          f"match_tiers={metrics['match_tier_counts']}")
    return metrics


In [11]:
# ================= RUN: all four cells of the 2x2 grid =================
# Load each adapter ONCE, run it against BOTH eval formats -- 4 evaluations total, all on the
# identical, audited 1622-utterance Session-5 set. No training anywhere below this line.
results = {}

print("="*78); print("Loading NoHist adapter (best checkpoint, epoch 2 of its training run)"); print("="*78)
base_model_a = load_fresh_base_model()
nohist_model = load_adapter_for_eval(base_model_a, NOHIST_ADAPTER_DIR)

print("\n--- A: NoHist-adapter x NoHist-eval (matched) ---")
results["A_nohist_adapter__nohist_eval"] = run_s5_eval(
    nohist_model, nohist_model.thinker, nohist_s5, tag="A_nohist_adapter__nohist_eval")

print("\n--- B: NoHist-adapter x Hist3-eval (mismatched) ---")
results["B_nohist_adapter__hist3_eval"] = run_s5_eval(
    nohist_model, nohist_model.thinker, hist3_s5, tag="B_nohist_adapter__hist3_eval")

del nohist_model, base_model_a
torch.cuda.empty_cache()

print("\n" + "="*78); print("Loading Hist3 adapter (best checkpoint, epoch 3 of its training run)"); print("="*78)
base_model_b = load_fresh_base_model()
hist3_model = load_adapter_for_eval(base_model_b, HIST3_ADAPTER_DIR)

print("\n--- C: Hist3-adapter x NoHist-eval (mismatched) ---")
results["C_hist3_adapter__nohist_eval"] = run_s5_eval(
    hist3_model, hist3_model.thinker, nohist_s5, tag="C_hist3_adapter__nohist_eval")

print("\n--- D: Hist3-adapter x Hist3-eval (matched) ---")
results["D_hist3_adapter__hist3_eval"] = run_s5_eval(
    hist3_model, hist3_model.thinker, hist3_s5, tag="D_hist3_adapter__hist3_eval")

del hist3_model, base_model_b
torch.cuda.empty_cache()

print("\nAll four Session-5 evaluations complete. No training occurred at any point above.")


Loading NoHist adapter (best checkpoint, epoch 2 of its training run)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

  loaded adapter from /kaggle/input/notebooks/pranavjaiganesh13/nohist-run/phase3_train_nohist_full/adapter_best: 7,372,800 LoRA params across 144 target modules, 0 trainable (inference-only), model.eval()

--- A: NoHist-adapter x NoHist-eval (matched) ---
    A_nohist_adapter__nohist_eval: 200/1622  (102s elapsed, 0.51s/example)
    A_nohist_adapter__nohist_eval: 400/1622  (203s elapsed, 0.51s/example)
    A_nohist_adapter__nohist_eval: 600/1622  (301s elapsed, 0.50s/example)
    A_nohist_adapter__nohist_eval: 800/1622  (401s elapsed, 0.50s/example)
    A_nohist_adapter__nohist_eval: 1000/1622  (498s elapsed, 0.50s/example)
    A_nohist_adapter__nohist_eval: 1200/1622  (596s elapsed, 0.50s/example)
    A_nohist_adapter__nohist_eval: 1400/1622  (696s elapsed, 0.50s/example)
    A_nohist_adapter__nohist_eval: 1600/1622  (800s elapsed, 0.50s/example)
  S5[A_nohist_adapter__nohist_eval]: WA=63.07  UA=59.183  macro_f1=60.188  weighted_f1=62.649  match_tiers={'exact': 1622, 'word_match': 0,

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  loaded adapter from /kaggle/input/notebooks/pranavjaiganesh13/hist3-run/phase3_train_hist3_full/adapter_best: 7,372,800 LoRA params across 144 target modules, 0 trainable (inference-only), model.eval()

--- C: Hist3-adapter x NoHist-eval (mismatched) ---
    C_hist3_adapter__nohist_eval: 200/1622  (96s elapsed, 0.48s/example)
    C_hist3_adapter__nohist_eval: 400/1622  (193s elapsed, 0.48s/example)
    C_hist3_adapter__nohist_eval: 600/1622  (288s elapsed, 0.48s/example)
    C_hist3_adapter__nohist_eval: 800/1622  (386s elapsed, 0.48s/example)
    C_hist3_adapter__nohist_eval: 1000/1622  (479s elapsed, 0.48s/example)
    C_hist3_adapter__nohist_eval: 1200/1622  (571s elapsed, 0.48s/example)
    C_hist3_adapter__nohist_eval: 1400/1622  (666s elapsed, 0.48s/example)
    C_hist3_adapter__nohist_eval: 1600/1622  (765s elapsed, 0.48s/example)
  S5[C_hist3_adapter__nohist_eval]: WA=61.406  UA=58.286  macro_f1=57.786  weighted_f1=60.381  match_tiers={'exact': 1622, 'word_match': 0, 'edit_di

In [12]:
# ================= 2x2 table + paired comparisons (training-context vs inference-context) =================
A = results["A_nohist_adapter__nohist_eval"]
B = results["B_nohist_adapter__hist3_eval"]
C = results["C_hist3_adapter__nohist_eval"]
D = results["D_hist3_adapter__hist3_eval"]

print("="*78)
print("2x2 TABLE  (rows = training adapter, cols = inference-time input format)")
print("="*78)
print(f"{'':22s}{'NoHist-eval':>16s}{'Hist3-eval':>16s}")
print(f"{'NoHist-adapter':22s}{A['macro_f1']:>16.3f}{B['macro_f1']:>16.3f}   (macro-F1)")
print(f"{'Hist3-adapter':22s}{C['macro_f1']:>16.3f}{D['macro_f1']:>16.3f}   (macro-F1)")
print()
print(f"{'':22s}{'NoHist-eval':>16s}{'Hist3-eval':>16s}")
print(f"{'NoHist-adapter':22s}{A['accuracy_WA']:>16.3f}{B['accuracy_WA']:>16.3f}   (WA)")
print(f"{'Hist3-adapter':22s}{C['accuracy_WA']:>16.3f}{D['accuracy_WA']:>16.3f}   (WA)")

print("\n" + "="*78)
print("PAIRED COMPARISONS  (all four share the identical 1622 S5 ids/order -- audited above)")
print("="*78)

print("\n--- Inference-context effect (does switching the EVAL format matter, holding the")
print("    training adapter fixed?) ---")
cmp_infer_nohist_adapter = EVAL.paired_comparison(
    SHARED_S5_TARGET_IDS, A, SHARED_S5_TARGET_IDS, B)
print(f"  NoHist-adapter, NoHist-eval -> Hist3-eval:  "
      f"delta_macro_f1={cmp_infer_nohist_adapter['delta_macro_f1']:+.3f}  "
      f"only_NoHist-eval_correct={cmp_infer_nohist_adapter['only_control_correct']}  "
      f"only_Hist3-eval_correct={cmp_infer_nohist_adapter['only_treatment_correct']}")
cmp_infer_hist3_adapter = EVAL.paired_comparison(
    SHARED_S5_TARGET_IDS, C, SHARED_S5_TARGET_IDS, D)
print(f"  Hist3-adapter,  NoHist-eval -> Hist3-eval:  "
      f"delta_macro_f1={cmp_infer_hist3_adapter['delta_macro_f1']:+.3f}  "
      f"only_NoHist-eval_correct={cmp_infer_hist3_adapter['only_control_correct']}  "
      f"only_Hist3-eval_correct={cmp_infer_hist3_adapter['only_treatment_correct']}")

print("\n--- Training-context effect (does switching WHICH adapter was trained matter,")
print("    holding the eval format fixed?) ---")
cmp_train_nohist_eval = EVAL.paired_comparison(
    SHARED_S5_TARGET_IDS, A, SHARED_S5_TARGET_IDS, C)
print(f"  NoHist-eval, NoHist-adapter -> Hist3-adapter:  "
      f"delta_macro_f1={cmp_train_nohist_eval['delta_macro_f1']:+.3f}  "
      f"only_NoHist-adapter_correct={cmp_train_nohist_eval['only_control_correct']}  "
      f"only_Hist3-adapter_correct={cmp_train_nohist_eval['only_treatment_correct']}")
cmp_train_hist3_eval = EVAL.paired_comparison(
    SHARED_S5_TARGET_IDS, B, SHARED_S5_TARGET_IDS, D)
print(f"  Hist3-eval,  NoHist-adapter -> Hist3-adapter:  "
      f"delta_macro_f1={cmp_train_hist3_eval['delta_macro_f1']:+.3f}  "
      f"only_NoHist-adapter_correct={cmp_train_hist3_eval['only_control_correct']}  "
      f"only_Hist3-adapter_correct={cmp_train_hist3_eval['only_treatment_correct']}")

print("\n--- Interaction: does the inference-context effect depend on the training adapter? ---")
interaction_v1 = cmp_infer_hist3_adapter["delta_macro_f1"] - cmp_infer_nohist_adapter["delta_macro_f1"]
interaction_v2 = cmp_train_hist3_eval["delta_macro_f1"] - cmp_train_nohist_eval["delta_macro_f1"]
print(f"  (D-C)-(B-A) = {interaction_v1:+.3f}   (D-B)-(C-A) = {interaction_v2:+.3f}   "
      f"(these are algebraically the same quantity -- {'MATCH' if abs(interaction_v1-interaction_v2)<0.01 else 'MISMATCH -- BUG'})")
if abs(interaction_v1 - interaction_v2) >= 0.01:
    print("  WARNING: interaction terms disagree by >0.01 -- investigate before interpreting.")

summary = {
    "cell_A_nohist_adapter__nohist_eval": {k: v for k, v in A.items() if k not in ("predictions",)},
    "cell_B_nohist_adapter__hist3_eval": {k: v for k, v in B.items() if k not in ("predictions",)},
    "cell_C_hist3_adapter__nohist_eval": {k: v for k, v in C.items() if k not in ("predictions",)},
    "cell_D_hist3_adapter__hist3_eval": {k: v for k, v in D.items() if k not in ("predictions",)},
    "inference_context_effect_nohist_adapter": cmp_infer_nohist_adapter,
    "inference_context_effect_hist3_adapter": cmp_infer_hist3_adapter,
    "training_context_effect_nohist_eval": cmp_train_nohist_eval,
    "training_context_effect_hist3_eval": cmp_train_hist3_eval,
    "interaction_delta_macro_f1": interaction_v1,
    "adapter_dirs": {"nohist": NOHIST_ADAPTER_DIR, "hist3": HIST3_ADAPTER_DIR},
    "n_s5_targets": len(SHARED_S5_TARGET_IDS),
}
summary_path = os.path.join(OUTPUT_DIR, "s5_2x2_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)
print(f"\nwrote {summary_path}")
print("\nSave Version -> Save & Run All (Commit) so all outputs under OUTPUT_DIR survive for review.")


2x2 TABLE  (rows = training adapter, cols = inference-time input format)
                           NoHist-eval      Hist3-eval
NoHist-adapter                  60.188          61.801   (macro-F1)
Hist3-adapter                   57.786          66.781   (macro-F1)

                           NoHist-eval      Hist3-eval
NoHist-adapter                  63.070          64.735   (WA)
Hist3-adapter                   61.406          68.804   (WA)

PAIRED COMPARISONS  (all four share the identical 1622 S5 ids/order -- audited above)

--- Inference-context effect (does switching the EVAL format matter, holding the
    training adapter fixed?) ---
  NoHist-adapter, NoHist-eval -> Hist3-eval:  delta_macro_f1=+1.613  only_NoHist-eval_correct=65  only_Hist3-eval_correct=92
  Hist3-adapter,  NoHist-eval -> Hist3-eval:  delta_macro_f1=+8.995  only_NoHist-eval_correct=98  only_Hist3-eval_correct=218

--- Training-context effect (does switching WHICH adapter was trained matter,
    holding the eval for

## Kaggle instructions

1. **Add Input**: the Session-5 dataset (`iemocap-ses5-6class` / `kaggle_upload/` -- the same
   one used for the zero-shot baseline and Experiments 2-3, containing `test.json` +
   `test_history3.json` + `audio/`).
2. **Add Input** (x2): the two COMPLETED training notebooks' outputs -- on Kaggle these are
   named `nohist_run` and `hist3_run`:
   - `nohist_run` -- the committed NoHist full-training run, containing
     `phase3_train_nohist_full/adapter_best/`
   - `hist3_run` -- the committed Hist3 full-training run, containing
     `phase3_train_hist3_full/adapter_best/`

   Use **Add Input -> search your own Notebooks** (not Datasets) and select each by name --
   Kaggle mounts a notebook's own `/kaggle/working/` output under `/kaggle/input/<that
   notebook>/` for any notebook that references it as an input. If auto-discovery in the
   dataset-location cell can't find them (prints a clear error naming what it searched for),
   set `NOHIST_ADAPTER_DIR_OVERRIDE` / `HIST3_ADAPTER_DIR_OVERRIDE` in the CONFIG cell to the
   exact path shown under `/kaggle/input/...` in the notebook's own file browser.
3. Settings: GPU T4 x2, Internet On.
4. **Save Version -> Save & Run All (Commit)** -- runs all four evaluations end-to-end in one
   go (no config flags to flip, no restart needed partway through).
5. All four predictions/metrics, plus `s5_2x2_summary.json`, land under
   `/kaggle/working/phase3_s5_2x2eval/`.
